In [0]:
%python
# %sql
# show catalogs

In [0]:
%python
# %sql
# CREATE VOLUME workspace.default.s3_volume COMMENT 'UC volume for file copies';

In [0]:
-- %run "/Workspace/Users/alban.berisha@inspire11.com/Databricks-Certified-Data-Engineer-Associate/Includes/Copy-Datasets"

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE orders_raw
COMMENT "The raw books orders"
SELECT * from STREAM read_files('/Volumes/workspace/default/s3_volume/bookstore/orders-raw', format => 'parquet')
  

In [0]:
CREATE OR REFRESH LIVE TABLE customers
COMMENT 'The customers lookup table'
SELECT *
from JSON.`/Volumes/workspace/default/s3_volume/bookstore/customers-json/`

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE orders_cleaned (
  CONSTRAINT valid_order_number EXPECT (order_id IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "The cleaned books orders with valid order_id"
AS
  SELECT order_id, quantity, o.customer_id, c.profile:first_name as f_name, c.profile:last_name as l_name,
         cast(from_unixtime(order_timestamp, 'yyyy-MM-dd HH:mm:ss') AS timestamp) order_timestamp, o.books,
         c.profile:address:country as country
  FROM STREAM(LIVE.orders_raw) o
  LEFT JOIN LIVE.customers c
    ON o.customer_id = c.customer_id

In [0]:
CREATE OR REFRESH LIVE TABLE cn_daily_customer_books
COMMENT "Daily number of books per customer in China"
AS
  SELECT customer_id, f_name, l_name, date_trunc("DD", order_timestamp) order_date, sum(quantity) books_counts
  FROM LIVE.orders_cleaned
  WHERE country = "China"
  GROUP BY customer_id, f_name, l_name, date_trunc("DD", order_timestamp)

#CDC

In [0]:
CREATE OR REPLACE STREAMING LIVE TABLE books_raw
select *
from STREAM read_files('/Volumes/workspace/default/s3_volume/bookstore/books-cdc',format => 'json')

In [0]:
CREATE OR REFRESH STREAMING TABLE books_silver;

APPLY CHANGES INTO LIVE.books_silver
FROM STREAM(LIVE.books_raw) 
keys(book_id)
APPLY AS DELETE WHEN row_status='DELETE'
SEQUENCE BY row_time
COLUMNS * EXCEPT (row_status,row_time)

In [0]:
--this is a live table but not a streaming table
CREATE LIVE TABLE author_counts_gold
AS SELECT author, count(*) as books_count, current_timestamp() as last_updated
FROM LIVE.books_silver
GROUP BY author

In [0]:
--this is a DLT view
CREATE LIVE VIEW books_sales
AS SELECT b.title,o.quantity
FROM (select *, explode(books) as book
from LIVE.orders_cleaned) o
INNER JOIN live.books_silver b
ON o.book.book_id = b.book_id;